# NQ ICT 1-Minute Backtester

This notebook is based on the uploaded Pine strategy: **15M bias → 5M liquidity → 1M sweep → MSS → displacement → FVG → retracement entry**.

**Important:** use real 1-minute NQ futures OHLC data. The backtester does not predict future prices and does not guarantee profitability.

In [ ]:

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

print("Backtester loaded.")


## 1. Load your NQ 1-minute CSV

Upload/export a CSV containing time, open, high, low, close (volume optional). Then set `DATA_FILE` below to the filename.

In [ ]:

# Upload the CSV.
# In Binder/Jupyter, change DATA_FILE to your uploaded filename.
DATA_FILE = ""

if DATA_FILE:
    raw = pd.read_csv(DATA_FILE)
else:
    from pathlib import Path
    csvs = sorted(Path(".").glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(
            "Upload a 1-minute NQ CSV into this notebook environment, "
            "then set DATA_FILE to its filename and run this cell again."
        )
    print("CSV files found:", [x.name for x in csvs])
    raw = pd.read_csv(csvs[0])

print("Columns:", list(raw.columns))
display(raw.head())


In [ ]:

# Normalize common CSV column names.
lower = {c.lower().strip(): c for c in raw.columns}

def find_col(names, required=True):
    for name in names:
        if name in lower:
            return lower[name]
    if required:
        raise ValueError(f"Missing one of {names}. Found: {list(raw.columns)}")
    return None

time_col = find_col(["time", "datetime", "date", "timestamp"])
open_col = find_col(["open"])
high_col = find_col(["high"])
low_col = find_col(["low"])
close_col = find_col(["close"])
vol_col = find_col(["volume", "vol"], required=False)

df = pd.DataFrame({
    "time": pd.to_datetime(raw[time_col], errors="coerce"),
    "open": pd.to_numeric(raw[open_col], errors="coerce"),
    "high": pd.to_numeric(raw[high_col], errors="coerce"),
    "low": pd.to_numeric(raw[low_col], errors="coerce"),
    "close": pd.to_numeric(raw[close_col], errors="coerce"),
})

df["volume"] = pd.to_numeric(raw[vol_col], errors="coerce") if vol_col else np.nan
df = df.dropna(subset=["time","open","high","low","close"]).sort_values("time").drop_duplicates("time")

if df["time"].dt.tz is None:
    df["time"] = df["time"].dt.tz_localize("America/New_York")
else:
    df["time"] = df["time"].dt.tz_convert("America/New_York")

df = df.set_index("time")
print(f"{len(df):,} one-minute bars")
print("From:", df.index.min())
print("To:  ", df.index.max())


## 2. Strategy settings

The defaults mirror the main inputs in your Pine code.

In [ ]:

CFG = {
    "bias_fast_len": 20,
    "bias_slow_len": 50,
    "bias_swing_len": 3,
    "liq_pivot_len": 3,
    "sweep_buffer_ticks": 0.0,
    "mss_lookback": 8,
    "displacement_atr": 0.80,
    "atr_length": 14,
    "setup_expiry": 10,
    "fvg_expiry": 12,
    "fvg_entry_mode": "50%",       # Touch, 50%, Full Fill
    "contracts": 1,
    "point_value": 20.0,
    "stop_buffer_ticks": 2.0,
    "minimum_rr": 1.50,
    "target_mode": "Liquidity",    # Liquidity, Fixed R, Hybrid
    "fixed_rr": 2.0,
    "move_to_be": True,
    "be_trigger_r": 1.0,
    "max_trades_day": 3,
    "max_losses_day": 2,
    "allow_longs": True,
    "allow_shorts": True,
    "session_start": "09:30",
    "session_end": "12:00",
    "tick_size": 0.25,
    "commission_per_contract_side": 2.50,
    "slippage_ticks": 1,
}
CFG


In [ ]:

def ohlc_resample(x, rule):
    return x.resample(rule, label="right", closed="right").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    }).dropna(subset=["open","high","low","close"])

def confirmed_pivot_high(s, n):
    candidate = s["high"].eq(s["high"].rolling(2*n+1, center=True).max())
    # Pivot is usable only after n bars have completed.
    return s["high"].where(candidate).shift(n)

def confirmed_pivot_low(s, n):
    candidate = s["low"].eq(s["low"].rolling(2*n+1, center=True).min())
    return s["low"].where(candidate).shift(n)

m5 = ohlc_resample(df, "5min")
m15 = ohlc_resample(df, "15min")

m15["ema_fast"] = m15["close"].ewm(span=CFG["bias_fast_len"], adjust=False).mean()
m15["ema_slow"] = m15["close"].ewm(span=CFG["bias_slow_len"], adjust=False).mean()
m15["ph"] = confirmed_pivot_high(m15, CFG["bias_swing_len"])
m15["pl"] = confirmed_pivot_low(m15, CFG["bias_swing_len"])

m5["ph"] = confirmed_pivot_high(m5, CFG["liq_pivot_len"])
m5["pl"] = confirmed_pivot_low(m5, CFG["liq_pivot_len"])

# Map completed HTF information onto each 1M bar.
work = df.copy()
for c in ["close","ema_fast","ema_slow","ph","pl"]:
    work["m15_" + c] = m15[c].reindex(work.index, method="ffill")

work["m5_ph"] = m5["ph"].reindex(work.index, method="ffill")
work["m5_pl"] = m5["pl"].reindex(work.index, method="ffill")

# Previous completed New York trading day high/low.
daily = df.resample("1D", label="right", closed="right").agg({"high":"max","low":"min"})
prev_daily = daily.shift(1)
work["prior_day_high"] = prev_daily["high"].reindex(work.index, method="ffill")
work["prior_day_low"] = prev_daily["low"].reindex(work.index, method="ffill")

# Wilder-style ATR approximation.
tr1 = work["high"] - work["low"]
tr2 = (work["high"] - work["close"].shift()).abs()
tr3 = (work["low"] - work["close"].shift()).abs()
work["tr"] = pd.concat([tr1,tr2,tr3], axis=1).max(axis=1)
work["atr"] = work["tr"].ewm(alpha=1/CFG["atr_length"], adjust=False).mean()

work["recent_swing_high"] = work["high"].shift(1).rolling(CFG["mss_lookback"]).max()
work["recent_swing_low"] = work["low"].shift(1).rolling(CFG["mss_lookback"]).min()

print("Prepared:", len(work), "1M bars")


## 3. Run the backtest

The engine processes the 1-minute candles chronologically and records entries, exits, P&L, R, and results.

In [ ]:

def run_backtest(data, cfg):
    tick = cfg["tick_size"]
    slip = cfg["slippage_ticks"] * tick
    trades = []

    long_armed = False
    short_armed = False
    long_sweep_bar = None
    short_sweep_bar = None
    long_sweep_low = np.nan
    short_sweep_high = np.nan

    long_fvg = None   # (bar_number, top, bottom)
    short_fvg = None

    position = None
    trades_today = 0
    losses_today = 0
    current_date = None

    # For 15M structure, keep the last two confirmed pivot values.
    bias_highs = []
    bias_lows = []

    last_buy_side = np.nan
    last_sell_side = np.nan

    rows = list(data.iterrows())

    for i, (ts, r) in enumerate(rows):
        if ts.date() != current_date:
            current_date = ts.date()
            trades_today = 0
            losses_today = 0

        if pd.notna(r["m5_ph"]):
            last_buy_side = float(r["m5_ph"])
        if pd.notna(r["m5_pl"]):
            last_sell_side = float(r["m5_pl"])

        # Update confirmed 15M pivots without using future information.
        # The ffill series can repeat a pivot across 1M bars, so update only
        # when the value changes.
        if pd.notna(r["m15_ph"]):
            if not bias_highs or r["m15_ph"] != bias_highs[-1]:
                bias_highs.append(float(r["m15_ph"]))
                bias_highs = bias_highs[-2:]
        if pd.notna(r["m15_pl"]):
            if not bias_lows or r["m15_pl"] != bias_lows[-1]:
                bias_lows.append(float(r["m15_pl"]))
                bias_lows = bias_lows[-2:]

        structure_bull = (
            len(bias_highs) == 2 and len(bias_lows) == 2
            and bias_highs[-1] > bias_highs[-2]
            and bias_lows[-1] > bias_lows[-2]
        )
        structure_bear = (
            len(bias_highs) == 2 and len(bias_lows) == 2
            and bias_highs[-1] < bias_highs[-2]
            and bias_lows[-1] < bias_lows[-2]
        )

        bull_bias = (
            pd.notna(r["m15_close"])
            and r["m15_close"] > r["m15_ema_fast"] > r["m15_ema_slow"]
            and structure_bull
        )
        bear_bias = (
            pd.notna(r["m15_close"])
            and r["m15_close"] < r["m15_ema_fast"] < r["m15_ema_slow"]
            and structure_bear
        )

        minutes = ts.hour * 60 + ts.minute
        start_m = int(cfg["session_start"][:2]) * 60 + int(cfg["session_start"][3:])
        end_m = int(cfg["session_end"][:2]) * 60 + int(cfg["session_end"][3:])
        in_session = start_m <= minutes <= end_m

        # ---------------- EXISTING POSITION ----------------
        if position is not None:
            side = position["side"]

            if side == "long":
                if cfg["move_to_be"] and not position["be"]:
                    if r["high"] >= position["entry"] + position["risk"] * cfg["be_trigger_r"]:
                        position["be"] = True

                stop = position["entry"] if position["be"] else position["stop"]
                hit_stop = r["low"] <= stop
                hit_target = r["high"] >= position["target"]

                if hit_stop or hit_target:
                    # Conservative same-bar assumption: stop first.
                    if hit_stop:
                        exit_price = stop - slip
                        result = "LOSS"
                    else:
                        exit_price = position["target"] - slip
                        result = "WIN"

                    gross = (exit_price - position["entry"]) * cfg["point_value"] * position["qty"]
                    commission = cfg["commission_per_contract_side"] * position["qty"] * 2
                    net = gross - commission

                    trades.append({
                        **position,
                        "exit_time": ts,
                        "exit": exit_price,
                        "gross_pnl": gross,
                        "commission": commission,
                        "net_pnl": net,
                        "r": (exit_price - position["entry"]) / position["risk"],
                        "result": "LOSS" if net < 0 else result,
                    })
                    if net < 0:
                        losses_today += 1
                    position = None
                    continue

            else:
                if cfg["move_to_be"] and not position["be"]:
                    if r["low"] <= position["entry"] - position["risk"] * cfg["be_trigger_r"]:
                        position["be"] = True

                stop = position["entry"] if position["be"] else position["stop"]
                hit_stop = r["high"] >= stop
                hit_target = r["low"] <= position["target"]

                if hit_stop or hit_target:
                    if hit_stop:
                        exit_price = stop + slip
                        result = "LOSS"
                    else:
                        exit_price = position["target"] + slip
                        result = "WIN"

                    gross = (position["entry"] - exit_price) * cfg["point_value"] * position["qty"]
                    commission = cfg["commission_per_contract_side"] * position["qty"] * 2
                    net = gross - commission

                    trades.append({
                        **position,
                        "exit_time": ts,
                        "exit": exit_price,
                        "gross_pnl": gross,
                        "commission": commission,
                        "net_pnl": net,
                        "r": (position["entry"] - exit_price) / position["risk"],
                        "result": "LOSS" if net < 0 else result,
                    })
                    if net < 0:
                        losses_today += 1
                    position = None
                    continue

        # ---------------- FILTERS ----------------
        if (
            position is not None
            or not in_session
            or trades_today >= cfg["max_trades_day"]
            or losses_today >= cfg["max_losses_day"]
        ):
            continue

        # ---------------- LIQUIDITY SWEEPS ----------------
        sweep_sell_side = (
            cfg["allow_longs"] and bull_bias
            and pd.notna(last_sell_side)
            and r["low"] < last_sell_side - cfg["sweep_buffer_ticks"] * tick
            and r["close"] > last_sell_side
        )
        sweep_buy_side = (
            cfg["allow_shorts"] and bear_bias
            and pd.notna(last_buy_side)
            and r["high"] > last_buy_side + cfg["sweep_buffer_ticks"] * tick
            and r["close"] < last_buy_side
        )
        sweep_prior_low = (
            cfg["allow_longs"] and bull_bias
            and pd.notna(r["prior_day_low"])
            and r["low"] < r["prior_day_low"] - cfg["sweep_buffer_ticks"] * tick
            and r["close"] > r["prior_day_low"]
        )
        sweep_prior_high = (
            cfg["allow_shorts"] and bear_bias
            and pd.notna(r["prior_day_high"])
            and r["high"] > r["prior_day_high"] + cfg["sweep_buffer_ticks"] * tick
            and r["close"] < r["prior_day_high"]
        )

        if sweep_sell_side or sweep_prior_low:
            long_armed = True
            long_sweep_bar = i
            long_sweep_low = float(r["low"])
            long_fvg = None

        if sweep_buy_side or sweep_prior_high:
            short_armed = True
            short_sweep_bar = i
            short_sweep_high = float(r["high"])
            short_fvg = None

        # ---------------- EXPIRE SWEEPS ----------------
        if long_armed and i - long_sweep_bar > cfg["setup_expiry"]:
            long_armed = False
            long_sweep_bar = None
            long_sweep_low = np.nan

        if short_armed and i - short_sweep_bar > cfg["setup_expiry"]:
            short_armed = False
            short_sweep_bar = None
            short_sweep_high = np.nan

        # ---------------- MSS / DISPLACEMENT ----------------
        bull_mss = (
            long_armed
            and pd.notna(r["recent_swing_high"])
            and r["close"] > r["recent_swing_high"]
            and r["close"] > r["open"]
        )
        bear_mss = (
            short_armed
            and pd.notna(r["recent_swing_low"])
            and r["close"] < r["recent_swing_low"]
            and r["close"] < r["open"]
        )

        body = abs(r["close"] - r["open"])
        bull_disp = bull_mss and body >= r["atr"] * cfg["displacement_atr"]
        bear_disp = bear_mss and body >= r["atr"] * cfg["displacement_atr"]

        # ---------------- FVG ----------------
        if i >= 2 and bull_disp and r["low"] > rows[i-2][1]["high"]:
            long_fvg = (i, float(r["low"]), float(rows[i-2][1]["high"]))
            long_armed = False

        if i >= 2 and bear_disp and r["high"] < rows[i-2][1]["low"]:
            short_fvg = (i, float(rows[i-2][1]["low"]), float(r["high"]))
            short_armed = False

        if long_fvg and i - long_fvg[0] > cfg["fvg_expiry"]:
            long_fvg = None
        if short_fvg and i - short_fvg[0] > cfg["fvg_expiry"]:
            short_fvg = None

        # ---------------- FVG RETRACEMENT ----------------
        candidates = []

        if long_fvg:
            _, top, bottom = long_fvg
            if cfg["fvg_entry_mode"] == "Touch":
                entry = top
            elif cfg["fvg_entry_mode"] == "50%":
                entry = (top + bottom) / 2
            else:
                entry = bottom

            reached = r["low"] <= entry and r["close"] >= bottom
            if reached:
                stop = long_sweep_low - cfg["stop_buffer_ticks"] * tick
                risk = entry - stop

                liquidity = np.nan
                if pd.notna(last_buy_side) and last_buy_side > entry:
                    liquidity = last_buy_side
                elif pd.notna(r["prior_day_high"]) and r["prior_day_high"] > entry:
                    liquidity = r["prior_day_high"]

                fixed_target = entry + risk * cfg["fixed_rr"]
                if cfg["target_mode"] == "Fixed R" or pd.isna(liquidity):
                    target = fixed_target
                elif cfg["target_mode"] == "Liquidity":
                    target = liquidity
                else:
                    target = liquidity if liquidity >= entry + risk * cfg["minimum_rr"] else fixed_target

                rr = (target - entry) / risk if risk > 0 else np.nan
                if risk > 0 and rr >= cfg["minimum_rr"]:
                    candidates.append(("long", entry, stop, target, risk, rr))

        if short_fvg:
            _, top, bottom = short_fvg
            if cfg["fvg_entry_mode"] == "Touch":
                entry = bottom
            elif cfg["fvg_entry_mode"] == "50%":
                entry = (top + bottom) / 2
            else:
                entry = top

            reached = r["high"] >= entry and r["close"] <= top
            if reached:
                stop = short_sweep_high + cfg["stop_buffer_ticks"] * tick
                risk = stop - entry

                liquidity = np.nan
                if pd.notna(last_sell_side) and last_sell_side < entry:
                    liquidity = last_sell_side
                elif pd.notna(r["prior_day_low"]) and r["prior_day_low"] < entry:
                    liquidity = r["prior_day_low"]

                fixed_target = entry - risk * cfg["fixed_rr"]
                if cfg["target_mode"] == "Fixed R" or pd.isna(liquidity):
                    target = fixed_target
                elif cfg["target_mode"] == "Liquidity":
                    target = liquidity
                else:
                    target = liquidity if liquidity <= entry - risk * cfg["minimum_rr"] else fixed_target

                rr = (entry - target) / risk if risk > 0 else np.nan
                if risk > 0 and rr >= cfg["minimum_rr"]:
                    candidates.append(("short", entry, stop, target, risk, rr))

        # ---------------- ENTER ----------------
        if candidates:
            side, entry, stop, target, risk, rr = candidates[0]
            qty = cfg["contracts"]

            position = {
                "side": side,
                "entry_time": ts,
                "entry": entry + (slip if side == "long" else -slip),
                "stop": stop,
                "target": target,
                "risk": risk,
                "rr_planned": rr,
                "qty": qty,
                "be": False,
            }
            trades_today += 1

            if side == "long":
                long_fvg = None
            else:
                short_fvg = None

    # Close an open trade at the last close so it appears in results.
    if position is not None:
        ts = data.index[-1]
        last = data.iloc[-1]["close"]
        if position["side"] == "long":
            gross = (last - position["entry"]) * cfg["point_value"] * position["qty"]
            r_mult = (last - position["entry"]) / position["risk"]
        else:
            gross = (position["entry"] - last) * cfg["point_value"] * position["qty"]
            r_mult = (position["entry"] - last) / position["risk"]
        commission = cfg["commission_per_contract_side"] * position["qty"] * 2
        net = gross - commission
        trades.append({
            **position, "exit_time": ts, "exit": last,
            "gross_pnl": gross, "commission": commission,
            "net_pnl": net, "r": r_mult,
            "result": "WIN" if net > 0 else "LOSS"
        })

    return pd.DataFrame(trades)

trades = run_backtest(work, CFG)
print("Completed trades:", len(trades))
if len(trades):
    display(trades.head(20))


## 4. Performance report

In [ ]:

if len(trades) == 0:
    print("No trades found. Try a larger date range or temporarily lower the minimum grade/R:R filters.")
else:
    wins = trades[trades["net_pnl"] > 0]
    losses = trades[trades["net_pnl"] <= 0]

    gross_profit = wins["net_pnl"].sum()
    gross_loss = abs(losses["net_pnl"].sum())
    profit_factor = gross_profit / gross_loss if gross_loss else np.inf
    win_rate = len(wins) / len(trades) * 100
    net_profit = trades["net_pnl"].sum()

    equity = trades["net_pnl"].cumsum()
    drawdown = equity - equity.cummax()
    max_dd = drawdown.min()

    print(f"Trades:        {len(trades)}")
    print(f"Win rate:      {win_rate:.2f}%")
    print(f"Profit factor: {profit_factor:.2f}")
    print(f"Net P&L:       ${net_profit:,.2f}")
    print(f"Avg trade:     ${trades['net_pnl'].mean():,.2f}")
    print(f"Avg R:         {trades['r'].mean():.2f}R")
    print(f"Max drawdown:  ${max_dd:,.2f}")
    print(f"Winners:       {len(wins)}")
    print(f"Losers:        {len(losses)}")

    display(
        trades[[
            "side","entry_time","entry","stop","target",
            "exit_time","exit","net_pnl","r","result"
        ]].sort_values("entry_time").reset_index(drop=True)
    )


## 5. Equity curve

In [ ]:

if len(trades):
    plt.figure(figsize=(12,5))
    plt.plot(trades["exit_time"], trades["net_pnl"].cumsum())
    plt.axhline(0, linewidth=1)
    plt.title("NQ ICT Backtest Equity Curve")
    plt.xlabel("Time")
    plt.ylabel("Cumulative Net P&L ($)")
    plt.grid(True, alpha=0.25)
    plt.show()


## 6. Daily breakdown

In [ ]:

if len(trades):
    by_day = trades.assign(day=trades["entry_time"].dt.date).groupby("day").agg(
        trades=("net_pnl","size"),
        pnl=("net_pnl","sum"),
        wins=("net_pnl", lambda x: (x > 0).sum()),
        avg_r=("r","mean")
    )
    by_day["win_rate"] = by_day["wins"] / by_day["trades"] * 100
    display(by_day.tail(30))
